In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
CONFIDENCE_THRESHOLD = 0.90
TARGET_ACCURACY = 0.85
MAX_ITERATIONS = 10





In [4]:
df = pd.read_csv("input_analysis_2.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)


In [5]:
# Target Label Creation
def label_movement(x):
    if x > 0.5:
        return "moved_up"
    elif x < -0.5:
        return "moved_down"
    else:
        return "sideways"

df["target"] = df["price_increase_percentage"].apply(label_movement)

# STRICT LEAKAGE PREVENTION
# 🚨 DO NOT allow same-day price info
LEAKAGE_COLUMNS = ["price_increase_percentage", "expected_result"]

for col in LEAKAGE_COLUMNS:
    if col in df.columns:
        df[col] = df[col].shift(1)


In [6]:
# 5. Feature Engineering (ALL NUMERIC, ALL LAGGED)
df["ret_1d"] = df["price_increase_percentage"].shift(1)
df["ret_2d"] = df["price_increase_percentage"].shift(2)
df["ret_3d"] = df["price_increase_percentage"].shift(3)

df["vol_5d"] = df["ret_1d"].rolling(5).std()
df["vol_20d"] = df["ret_1d"].rolling(20).std()
df["vol_ratio"] = df["vol_5d"] / (df["vol_20d"] + 1e-6)
df["vol_chg"] = df["volume"].pct_change()
df["oi_chg"] = df["open_interest"].pct_change()


In [7]:
# 6. Astrology → Numeric (FIXED & SAFE)

RAASHI_MAP = {
    "mesh":1,"vrishabh":2,"mithun":3,"karka":4,"simha":5,"kanya":6,
    "tula":7,"vrishchik":8,"dhanu":9,"makara":10,"kumbha":11,"meena":12
}

df["sun_raashi"] = df["b_sun_raashi_name"].map(RAASHI_MAP).fillna(0)
df["sun_element"] = ((df["sun_raashi"] - 1) // 3) % 4
df["sun_modality"] = df["sun_raashi"] % 3

TITHI_MAP = {
    "pratipada":1,"dvitiya":2,"tritiya":3,"chaturthi":4,"panchami":5,
    "shashthi":6,"saptami":7,"ashtami":8,"navami":9,"dashami":10,
    "ekadashi":11,"dwadashi":12,"trayodashi":13,"chaturdashi":14,
    "purnima":15,"amavasya":15
}

def parse_tithi(x):
    if pd.isna(x):
        return (0,0)
    parts = str(x).split("_")
    name = parts[0]
    phase = 1 if "shukla" in x or name=="purnima" else 0
    return (TITHI_MAP.get(name,0), phase)

df[["tithi_ord","tithi_phase"]] = pd.DataFrame(
    df["ca_tithi_name"].apply(parse_tithi).tolist(),
    index=df.index
)


In [8]:
# 7. Final Feature Set

FEATURES = [
    "ret_1d","ret_2d","ret_3d",
    "vol_5d","vol_20d","vol_ratio",
    "vol_chg","oi_chg",
    "sun_raashi","sun_element","sun_modality",
    "tithi_ord","tithi_phase"
]

df = df.dropna().reset_index(drop=True)
X = df[FEATURES]
y = df["target"]


In [9]:
# 8. Automated Iterative Training Loop (REAL)

best_accuracy = 0
best_model = None

for iteration in range(1, MAX_ITERATIONS+1):
    print(f"\n🔁 ITERATION {iteration}")

    tscv = TimeSeriesSplit(n_splits=5)
    preds, trues, confs = [], [], []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = GradientBoostingClassifier(random_state=RANDOM_STATE)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)
        confidence = proba.max(axis=1)
        pred = model.classes_[np.argmax(proba, axis=1)]

        pred = np.where(confidence < CONFIDENCE_THRESHOLD, "sideways", pred)

        preds.extend(pred)
        trues.extend(y_test.values)
        confs.extend(confidence)

    acc = accuracy_score(trues, preds)
    print(f"Accuracy: {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

    if acc >= TARGET_ACCURACY:
        print("✅ TARGET ACHIEVED")
        break



🔁 ITERATION 1
Accuracy: 0.4389

🔁 ITERATION 2
Accuracy: 0.4389

🔁 ITERATION 3
Accuracy: 0.4389

🔁 ITERATION 4
Accuracy: 0.4389

🔁 ITERATION 5
Accuracy: 0.4389

🔁 ITERATION 6
Accuracy: 0.4389

🔁 ITERATION 7
Accuracy: 0.4389

🔁 ITERATION 8
Accuracy: 0.4389

🔁 ITERATION 9
Accuracy: 0.4389

🔁 ITERATION 10
Accuracy: 0.4389


In [10]:
# 9. Final Evaluation & Confidence Flag
results = pd.DataFrame({
    "actual": trues,
    "predicted": preds,
    "confidence": confs
})

results["LOW_CONFIDENCE"] = results["confidence"] < CONFIDENCE_THRESHOLD

print("\nConfusion Matrix:")
print(confusion_matrix(results["actual"], results["predicted"]))

print("\nClassification Report:")
print(classification_report(results["actual"], results["predicted"]))

print("\nCoverage:", (~results["LOW_CONFIDENCE"]).mean())



Confusion Matrix:
[[ 0  2 31]
 [ 0  3 63]
 [ 1  4 76]]

Classification Report:
              precision    recall  f1-score   support

  moved_down       0.00      0.00      0.00        33
    moved_up       0.33      0.05      0.08        66
    sideways       0.45      0.94      0.61        81

    accuracy                           0.44       180
   macro avg       0.26      0.33      0.23       180
weighted avg       0.32      0.44      0.30       180


Coverage: 0.28888888888888886


In [11]:
results["actual"]

0      moved_down
1      moved_down
2      moved_down
3        moved_up
4        moved_up
          ...    
175    moved_down
176      moved_up
177      sideways
178      sideways
179      moved_up
Name: actual, Length: 180, dtype: object

# Approach 2

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from xgboost import XGBClassifier

# 1. Load data
df = pd.read_csv("input_analysis_2.csv")
# Ensure datetime sorting if needed
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

# 2. Build target: 1 for up (expected_result==1), 0 for down (expected_result==2)
if 'expected_result' in df.columns:
    df['target'] = (df['expected_result'] == 1).astype(int)
else:
    raise ValueError("Expected result column not found.")

# 3. Compute any needed price features
if {'Open','Close'}.issubset(df.columns):
    df['pct_day'] = (df['Close'] - df['Open']) / df['Open'] * 100.0

# 4. Parse astrological fields if present
# Example: map Sun sign (Raashi) names to numeric index
raashi_map = {
    'mesh':1,'vrishabh':2,'mithun':3,'karka':4,'simha':5,'kanya':6,
    'tula':7,'vrishchik':8,'dhanu':9,'makara':10,'kumbha':11,'meena':12
}
if 'b_sun_raashi_name' in df.columns:
    df['sun_raashi_index'] = df['b_sun_raashi_name'].map(raashi_map).fillna(0).astype(int)

# Parse Tithi (ordinal + waxing flag)
tithi_map = {
    'pratipada':1,'dvitiya':2,'tritiya':3,'chaturthi':4,'panchami':5,
    'shashthi':6,'saptami':7,'ashtami':8,'navami':9,'dashami':10,
    'ekadashi':11,'dwadashi':12,'trayodashi':13,'chaturdashi':14,
    'purnima':15,'amavasya':15
}
def parse_tithi(name):
    if pd.isna(name): 
        return (0,0)
    parts = name.split('_')
    base = parts[0]
    cycle = parts[1] if len(parts)>1 else ''
    num = tithi_map.get(base, 0)
    shukla_flag = 1 if (cycle=='shukla' or base=='purnima') else 0
    return pd.Series([num, shukla_flag], index=['tithi_num','tithi_shukla'])

for col in ['ca_tithi_name','cb_tithi_name']:
    if col in df.columns:
        df[['tithi_num','tithi_shukla']] = df[col].apply(lambda x: parse_tithi(x))
        # If two columns (ca_, cb_), prefix them differently if needed

# Map Nakshatra to index (27 nakshatras)
nak_order = [
    'ashwini','bharani','krttika','rohini','mrugasira','ardra','punarbha',
    'pusya','aslesha','magha','purva_phalguni','uttara_phalguni','hasta',
    'chitra','swati','visakha','anuradha','jyeshta','moola','purva_asadha',
    'uttara_asadha','sravana','dhanishta','satabhisha','purva_bhadrapada',
    'uttara_bhadrapada','revati'
]
for col in ['da_nakshatra_name','db_nakshatra_name']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('_','')
        df[col] = df[col].str.lower()
        df[col+'_idx'] = df[col].apply(lambda x: nak_order.index(x)+1 if x in nak_order else 0)

# 5. Construct feature set
feature_cols = []
for name in ['Open','High','Low','Volume','pct_day']:
    if name in df.columns:
        feature_cols.append(name)
# Include astro-derived features if available
if 'sun_raashi_index' in df.columns:
    feature_cols.append('sun_raashi_index')
if 'tithi_num' in df.columns:
    feature_cols.extend(['tithi_num','tithi_shukla'])
for name in ['da_nakshatra_name_idx','db_nakshatra_name_idx']:
    if name in df.columns:
        feature_cols.append(name)

X = df[feature_cols].fillna(0)
y = df['target']

# 6. Train/Test split with scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, 
                                                    test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# 7. Train classifier
clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf.fit(X_train_s, y_train)

# 8. Evaluate accuracy
y_pred = clf.predict(X_test_s)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


AttributeError: 'builtin_function_or_method' object has no attribute 'get_indexer'